# Credit ratios from a statement model

Compute **leverage**, **coverage**, and **liquidity** from evaluated nodes, then layer `run_corporate_analysis`, `pl_summary_report`, and `credit_assessment_report` when available.

## Concept

Credit workpapers pair **P&L** (EBITDA, interest) with **balance sheet** (debt, cash). Statement nodes compute facility-defined metrics, while `covenants.evaluate_engine` owns threshold tests, pass/fail decisions, and headroom.

In [1]:
import json

from finstack_quant.covenants import evaluate_engine, lbo_standard, validate_covenant_engine
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import (
    credit_assessment,
    credit_assessment_report,
    pl_summary_report,
    run_corporate_analysis,
)

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]


def series(v):
    return list(zip(PERIODS, v))


b = ModelBuilder("credit-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.with_meta("currency", '"USD"')
b.value("revenue", series([200.0, 210.0, 218.0, 225.0]))
b.compute("ebitda", "revenue * 0.18")
b.value("interest_expense", series([7.0, 7.2, 7.5, 7.8]))
b.compute("ebit", "ebitda - interest_expense")
b.value("principal_paid", series([5.0, 5.0, 5.5, 5.5]))
b.value("total_debt", series([120.0, 118.0, 115.0, 112.0]))
b.value("cash", series([15.0, 16.0, 17.0, 18.0]))
b.compute("net_debt", "total_debt - cash")
b.value("capex", series([8.0, 8.5, 9.0, 9.0]))
b.compute("ufcf", "ebitda - capex")
b.compute("free_cash_flow", "ebitda - interest_expense - principal_paid - capex")
b.compute("debt_to_ebitda", "total_debt / ebitda")
b.compute("interest_coverage", "ebitda / interest_expense")
b.compute("fixed_charge_coverage", "(ebitda - capex) / (interest_expense + principal_paid)")

spec = b.build()
res = Evaluator().evaluate(spec)
eval_json = res.to_json()
model_json = spec.to_json()

as_of = "2025Q4"
covenant_specs = json.loads(
    lbo_standard(
        initial_leverage=3.0,
        interest_coverage=3.0,
        fixed_charge_coverage=1.5,
        max_capex=12.0,
    )
)
engine_json = json.dumps({"specs": covenant_specs, "breach_history": [], "windows": [], "waivers": []})
validate_covenant_engine(engine_json)
metrics = json.dumps({
    "debt_to_ebitda": res.get("debt_to_ebitda", as_of),
    "interest_coverage": res.get("interest_coverage", as_of),
    "fixed_charge_coverage": res.get("fixed_charge_coverage", as_of),
    "capex": res.get("capex", as_of),
})
covenant_report = json.loads(evaluate_engine(engine_json, metrics, "2025-12-31"))
for covenant_id, result in covenant_report.items():
    print(covenant_id, "passed=", result["passed"], "headroom=", result["headroom"])

print(pl_summary_report(eval_json, ["revenue", "ebitda", "interest_expense"], PERIODS))
print(credit_assessment_report(eval_json, as_of))

max_debt_ebitda passed= True headroom= 0.07818930041152268
min_interest_coverage passed= True headroom= 0.7307692307692308
min_fcc passed= True headroom= 0.5789473684210525
max_capex passed= True headroom= 0.25
P&L Summary

┌──────────────────┬────────┬────────┬────────┬────────┐
│ Line Item        │ 2025Q1 │ 2025Q2 │ 2025Q3 │ 2025Q4 │
├──────────────────┼────────┼────────┼────────┼────────┤
│ revenue          │ 200.00 │ 210.00 │ 218.00 │ 225.00 │
│ ebitda           │  36.00 │  37.80 │  39.24 │  40.50 │
│ interest_expense │   7.00 │   7.20 │   7.50 │   7.80 │
└──────────────────┴────────┴────────┴────────┴────────┘
Credit Assessment as of 2025Q4

Total Debt / TTM EBITDA:    0.73x
TTM EBITDA / TTM Interest:  5.20x
Free Cash Flow:             $0.00M



In [2]:
print("Corporate analysis + downside covenant snapshot")
tv = json.dumps({"type": "gordon_growth", "growth_rate": 0.02})
corp = run_corporate_analysis(
    spec,
    wacc=0.10,
    terminal_value_json=tv,
    net_debt_override=95.0,
    coverage_node="ebitda",
)
print("run_corporate_analysis keys:", sorted(corp.keys()))
if "equity" in corp:
    print("Equity value:", corp["equity"])

ca = credit_assessment(res, "2025Q4")
print("credit_assessment keys:", list(ca.keys()) if isinstance(ca, dict) else type(ca))

downside_metrics = json.dumps({
    "debt_to_ebitda": 3.4,
    "interest_coverage": 2.6,
    "fixed_charge_coverage": 1.3,
    "capex": 9.0,
})
downside_report = json.loads(evaluate_engine(engine_json, downside_metrics, "2025-12-31"))
print("Downside covenant results:")
for covenant_id, result in downside_report.items():
    print(covenant_id, "passed=", result["passed"], "headroom=", result["headroom"])

Corporate analysis + downside covenant snapshot
run_corporate_analysis keys: ['credit', 'equity', 'statement_json']
Equity value: {'equity_value': 1465.517526680174, 'equity_currency': 'USD', 'enterprise_value': 1560.517526680174, 'net_debt': 95.0, 'terminal_value_pv': 1500.956014302885, 'equity_value_per_share': None, 'diluted_shares': None}
credit_assessment keys: ['as_of', 'leverage_ratio', 'interest_coverage', 'free_cash_flow', 'series']
Downside covenant results:
max_debt_ebitda passed= False headroom= -0.1333333333333333
min_interest_coverage passed= False headroom= -0.1333333333333333
min_fcc passed= False headroom= -0.1333333333333333
max_capex passed= True headroom= 0.25


## Takeaways

- **Leverage and coverage** are statement nodes aligned to facility definitions.
- `credit_assessment_report` is a fast text view of the evaluated statements.
- `covenants.evaluate_engine` is the sole source for covenant pass/fail and headroom in both base and downside snapshots.